In [ ]:
from models_HyperNetwork import HyperNetworkSEModel
from generate_utils import load_GraphModel, load_BiLSTMModel, generate_files_with_nucleus
import torch
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer
from graph_utils import get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
device_name = 'cuda:0'
device = torch.device(device_name)

graph_model_path = 'saved_models/graph/graph_model.pt'
transformer_graph_path = 'saved_models/graph/transformer_model.pt'

bilstm_model_path = 'saved_models/bilstm/bilstm_model.pt'
transformer_bilstm_path = 'saved_models/bilstm/transformer_model.pt'

In [4]:
graph_model = load_GraphModel(graph_model_path, device)
bilstm_model = load_BiLSTMModel(bilstm_model_path, device)

graph_model.eval()
bilstm_model.eval()

HarmonyBiLSTM(
  (input_proj): Sequential(
    (0): Linear(in_features=12, out_features=64, bias=True)
    (1): ReLU()
  )
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (attn): Linear(in_features=256, out_features=1, bias=True)
  (output_proj): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
  )
)

In [5]:
datasets = {
    'gjt': {'path': 'data/gjt_test.pkl', 'dataset': None},
    # 'hook': {'path': 'data/hook_test.pkl', 'dataset': None},
    'nott': {'path': 'data/nott_test.pkl', 'dataset': None},
    # 'wiki': {'path': 'data/wiki_test.pkl', 'dataset': None}
}

for k, v in datasets.items():
    print(f'loading {k}')
    with open(v['path'], 'rb') as f:
        d = pickle.load(f)
    v['dataset'] = d

loading gjt
loading nott


In [6]:
in_seq = 'E:min;A:maj;F:min;A#:maj'

y_graph = get_graph_embeddings_from_string_with_model(in_seq, graph_model)
y_bilstm = get_bilstm_embeddings_from_string_with_model(in_seq, bilstm_model)

E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297
E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297


In [ ]:
transformer_model = HyperNetworkSEModel(
    len(tokenizer.vocab),
    graph_model.output_dim,
    device,
    d_model=512,
    nhead=8,
    num_layers=8,
    dim_feedforward=2048,
    pianoroll_dim=13,
    grid_length=80,
    dropout=0.3
)

transformer_model.to(device)

LoRAFiLMSEModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (hyper_q): HyperGuide(
    (layer_embedding): Embedding(8, 16)
    (head_embedding): Embedding(8, 16)
    (mlp): Sequential(
      (0): Linear(in_features=160, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): GELU(approximate='none')
    )
    (gamma_proj): Linear(in_features=128, out_features=64, bias=True)
    (beta_proj): Linear(in_features=128, out_features=64, bias=True)
    (adapter_proj): Linear(in_features=128, out_features=32, bias=True)
  )
  (hyper_k): HyperGuide(
    (layer_embedding): Embedding(8, 16)
    (head_embedding): Embedding(8, 16)
    (mlp): Sequential(
      (0): Linear(in_features=160, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=128, bias=True)
      (3): GELU(approximate='non

In [8]:
x = datasets['gjt']['dataset'][0]
print(x)

{'harmony_ids': [6, 217, 217, 217, 217, 6, 217, 217, 217, 217, 6, 194, 194, 194, 194, 6, 332, 332, 332, 332, 6, 131, 131, 131, 131, 6, 131, 131, 131, 131, 6, 80, 80, 80, 80, 6, 231, 231, 231, 231, 6, 14, 14, 14, 14, 6, 168, 168, 168, 168, 6, 217, 217, 217, 217, 6, 131, 131, 131, 131, 6, 284, 284, 284, 284, 6, 284, 284, 284, 284, 6, 276, 276, 276, 276, 6, 71, 71, 71, 71], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'pianoroll': array([[0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.]], dtype=float32), 'time_signature': [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], 'h_density_complexity': [1, 

In [9]:
y_graph = get_graph_embeddings_from_string_with_model(in_seq, graph_model)

E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297


In [10]:
y = transformer_model(
    torch.tensor(x['pianoroll'], dtype=torch.float32).unsqueeze(0).to(device),
    z_g=y_graph.unsqueeze(0)
)